# Threads

**Threads**: An entity within a process that can be scheduled (also known as "lightweight process").
A process can spawn multiple threads.

```
+ All threads within a process share the same memory
+ Lightweight
+ Starting a thread is faster than starting a process
+ Great for I/O bound tasks

- Threading is limited by GIL: Only one thread at a time
- No effect for CPU-bound tasks
- Not interruptable/killable
- Careful with race condition
```

# GIL: Global interpreter lock

```
- A lock that allows only one thread at a time to execute in Python
- Needed in CPython because memory management is not thread-safe

- Avoid:
  - Use multiprocessing
  - Use a different, free-threaded Python implementation (Jython, IronPython)
  - Use Python as a wrapper for third-party libraries (C/C++) -> numpy, scipy
```

In [1]:
from threading import Thread
import os
import time

def square_numbers():
  for i in range(100):
    i*i
    time.sleep(0.1)

if __name__ == '__main__':
  threads = []
  # threads_count = int(os.cpu_count()/2);
  threads_count = min(4,os.cpu_count()//2);

  for _ in range(threads_count):
    t = Thread(target=square_numbers)
    threads.append(t)

  for t in threads:
    t.start()

  for t in threads:
    t.join()

  print(f"End main.")


End main.


# Threading with shared object and locking

Because threads are spawned from same process, they all share same memory space.

So below code shows how to share data between threads using `global` variable.

In [3]:
from threading import Thread, Lock
import time

database_value = 0

def increase(lock):
  global database_value

  with lock:
    local_copy = database_value
    local_copy += 1
    time.sleep(0.1)
    database_value = local_copy

if __name__ == '__main__':

  lock = Lock()
  print(f"Value at start: {database_value}")

  t1 = Thread(target=increase, args=(lock,))
  t2 = Thread(target=increase, args=(lock,))

  t1.start()
  t2.start()

  t1.join()
  t2.join()

  print(f"Value at end: {database_value}")


Value at start: 0
Value at end: 2


# Using Queue with threading

`Queue` are excelling for thread safe & process safe data exchanges in a multi-processing & multi-threading environment.

It is a linear data structure that follows **First-In-First-Out** principle.

In [6]:
from threading import Thread, Lock, current_thread
from queue import Queue
import time

def worker(q, lock):
  while True:
    value = q.get()
    with lock:
      print(f"in {current_thread().name} got {value}")
    q.task_done()


if __name__=='__main__':
  q = Queue()
  lock = Lock()
  num_threads = 10

  for i in range(num_threads):
    thread = Thread(target=worker, args=(q, lock))
    thread.daemon = True
    thread.start()

  for i in range(1,21):
    q.put(i)

  q.join()

  print('end main')


in Thread-33 (worker) got 1
in Thread-33 (worker) got 2
in Thread-33 (worker) got 3
in Thread-33 (worker) got 4
in Thread-33 (worker) got 5
in Thread-33 (worker) got 6
in Thread-33 (worker) got 7
in Thread-33 (worker) got 8
in Thread-33 (worker) got 9
in Thread-33 (worker) got 10
in Thread-33 (worker) got 11
in Thread-33 (worker) got 12
in Thread-33 (worker) got 13
in Thread-33 (worker) got 14
in Thread-33 (worker) got 15
in Thread-33 (worker) got 16
in Thread-33 (worker) got 17
in Thread-33 (worker) got 18
in Thread-33 (worker) got 19
in Thread-33 (worker) got 20
end main
